# Conflict 1: Engagement Scoring vs. User Wellbeing
## Ethical Framework: Utilitarianism

X/Twitter's algorithm ranks tweets by predicted engagement (likes, retweets, replies). The functional goal is to promote the most engaging content.

The **utilitarian ethical constraint** says: maximise total user wellbeing, which means harmful content should not be promoted even if engagement is high.

### The Tension
Short-term engagement ≠ long-term wellbeing. Outrage-bait and misinformation generate high engagement but reduce total user welfare. A utilitarian analysis demands we optimise for the *right* metric.

## Setup
Install Z3 if needed and import dependencies.

In [3]:
%pip install z3-solver
from z3 import *

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Domain Model

We model 3 candidate tweets competing for promotion in the For You feed. Each tweet has:
- An **engagement score** (predicted likes + retweets + replies)
- A **harm flag** (misinformation, outrage-bait, mental health damage)
- A **promoted** status (whether the algorithm shows it)

In [4]:
s = Solver()

# Domain: 3 candidate tweets
engagement = [Int(f'engagement_{i}') for i in range(3)]
harmful    = [Bool(f'harmful_{i}') for i in range(3)]
promoted   = [Bool(f'promoted_{i}') for i in range(3)]

# Tweet 0: viral outrage-bait (high engagement, harmful)
s.add(engagement[0] == 95)
s.add(harmful[0] == True)

# Tweet 1: informative news article (medium engagement, not harmful)
s.add(engagement[1] == 60)
s.add(harmful[1] == False)

# Tweet 2: casual personal post (low engagement, not harmful)
s.add(engagement[2] == 30)
s.add(harmful[2] == False)

print("Scenario:")
print("  Tweet 0: engagement=95, HARMFUL (outrage-bait)")
print("  Tweet 1: engagement=60, safe (news article)")
print("  Tweet 2: engagement=30, safe (personal post)")

Scenario:
  Tweet 0: engagement=95, HARMFUL (outrage-bait)
  Tweet 1: engagement=60, safe (news article)
  Tweet 2: engagement=30, safe (personal post)


## Functional Goal: Promote Highest Engagement

The algorithm must promote exactly one tweet — the one with the highest engagement score. This models X's core ranking behaviour: the 48M parameter neural network optimises for predicted engagement.

In [5]:
# Functional Goal: promote the single highest-engagement tweet
s.assert_and_track(
    Or([
        And(promoted[i],
            And([engagement[i] >= engagement[j] for j in range(3)]),
            And([Not(promoted[j]) for j in range(3) if j != i]))
        for i in range(3)
    ]),
    "FUNCTIONAL_promote_highest_engagement"
)

# At least one tweet must be promoted
s.assert_and_track(
    Or(promoted[0], promoted[1], promoted[2]),
    "FUNCTIONAL_must_promote_something"
)

print("Functional constraints added:")
print("  - Must promote the tweet with highest engagement score")
print("  - Exactly one tweet is promoted")

Functional constraints added:
  - Must promote the tweet with highest engagement score
  - Exactly one tweet is promoted


## Ethical Goal: Never Promote Harmful Content (Utilitarian)

A utilitarian analysis shows that harmful content reduces total user wellbeing even when engagement is high. The aggregate "good" (wellbeing) is not the same as the aggregate "clicks" (engagement).

We formalise this as: `harmful(t) → ¬promoted(t)` for all tweets.

In [6]:
# Ethical Goal: never promote harmful content
for i in range(3):
    s.assert_and_track(
        Implies(harmful[i], Not(promoted[i])),
        f"ETHICAL_no_promote_harmful_tweet_{i}"
    )

print("Ethical constraints added:")
print("  - If a tweet is harmful, it must not be promoted")

Ethical constraints added:
  - If a tweet is harmful, it must not be promoted


## Conflict Detection

We now check whether both goals can be satisfied simultaneously.

In [8]:
print("CONFLICT 1: Engagement Scoring vs. Wellbeing")
print("Ethical Framework: Utilitarianism")

result = s.check()

if result == sat:
    print("\nResult: SAT (No conflict)")
    m = s.model()
    for i in range(3):
        status = "PROMOTED" if m.evaluate(promoted[i]) else "not promoted"
        harm = "HARMFUL" if m.evaluate(harmful[i]) else "safe"
        print(f"  Tweet {i}: engagement={m.evaluate(engagement[i])}, "
              f"{harm}, {status}")

elif result == unsat:
    print("\nResult: UNSAT — Conflict detected!")
    core = s.unsat_core()
    print(f"\nUnsat Core ({len(core)} conflicting constraints):")
    for c in core:
        print(f"  - {c}")

    print("\nInterpretation:")
    print("  The functional goal (promote highest engagement) conflicts")
    print("  with the ethical goal (never promote harmful content) because")
    print("  the most engaging tweet (Tweet 0, score=95) is also harmful.")

CONFLICT 1: Engagement Scoring vs. Wellbeing
Ethical Framework: Utilitarianism

Result: UNSAT — Conflict detected!

Unsat Core (2 conflicting constraints):
  - FUNCTIONAL_promote_highest_engagement
  - ETHICAL_no_promote_harmful_tweet_0

Interpretation:
  The functional goal (promote highest engagement) conflicts
  with the ethical goal (never promote harmful content) because
  the most engaging tweet (Tweet 0, score=95) is also harmful.


## Resolution Options

To resolve the UNSAT state, we must relax one constraint:

1. **Relax functional goal** — promote the highest-engagement tweet that is NOT harmful (engagement-with-guardrails)
2. **Relax ethical goal** — allow harmful content if engagement exceeds a threshold (risk-tolerant utilitarianism)
3. **Redefine engagement** — subtract a harm penalty from the engagement score (wellbeing-adjusted engagement)